# 03 Validation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.chdir("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/")

In [3]:
from src.validator import Validator
from src.output import load_config

In [4]:
from __future__ import annotations
import logging
import sys
from pathlib import Path

import pandas as pd
import yaml

In [5]:

def _build_logger(name: str = "Urban_Validator") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        fmt = logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        )
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        logger.addHandler(sh)
    logger.setLevel(logging.INFO)
    return logger

In [6]:
def load_tracker(tracker_path: str | Path) -> pd.DataFrame:
    """
    Load the AOI tracker CSV with normalised column names and stripped strings.

    Strips leading/trailing whitespace from every string cell — the CSV has
    several entries with accidental padding (e.g. " rohingya_4604_hotosm.geojson").
    """
    df = pd.read_csv(tracker_path, dtype=str)
    df.columns = df.columns.str.strip()          # strip BOM / header padding
    df = df.apply(lambda col: col.str.strip() if col.dtype == object else col)
    return df

def _is_truthy(val: str) -> bool:
    return str(val).strip().lower() in ("true", "yes", "1")

def resolve_sub_areas(
    group: pd.DataFrame,
    data_dir: Path,
    logger: logging.Logger,
) -> list[dict]:
    """
    Turn a group of CSV rows (all sharing the same Dataset code) into a list
    of validated sub-area dicts ready for Validator.

    Each returned dict has:
        sub_area_id : str   – derived from aoi_file_name stem
        aoi         : Path  – resolved, confirmed to exist
        reference   : Path  – resolved, confirmed to exist
    """
    sub_areas = []

    for _, row in group.iterrows():
        city         = row["Dataset code"]
        folder       = row["dataset_folder_name"]
        aoi_file     = row.get("aoi_file_name", "").strip()
        ref_file     = row.get("reference_file_name", "").strip()
        has_aoi      = _is_truthy(row.get("has_aoi_file", "false"))
        has_ref      = _is_truthy(row.get("has_reference_file", "false"))

        # Skip rows that are known incomplete
        if not has_aoi:
            logger.warning("  Skipping row (no AOI file): %s / %s", city, ref_file or "?")
            continue
        if not has_ref:
            logger.warning("  Skipping row (no reference file): %s / %s", city, aoi_file or "?")
            continue
        if not aoi_file:
            logger.warning("  Skipping row (empty aoi_file_name): %s", city)
            continue
        if not ref_file:
            logger.warning("  Skipping row (empty reference_file_name): %s", city)
            continue

        # Build paths from the convention
        aoi_path = data_dir / folder / "aoi"    / aoi_file
        ref_path = data_dir / folder / "vector" / ref_file

        # Hard-check existence so problems surface immediately
        if not aoi_path.exists():
            logger.error("  AOI file not found, skipping: %s", aoi_path)
            continue
        if not ref_path.exists():
            logger.error("  Reference file not found, skipping: %s", ref_path)
            continue

        # Use the AOI filename stem as the sub-area identifier
        # e.g. "jubacitycenter_aoi.geojson" → "jubacitycenter"
        sub_area_id = Path(aoi_file).stem.replace("_aoi", "")

        sub_areas.append({
            "sub_area_id": sub_area_id,
            "aoi":         aoi_path,
            "reference":   ref_path,
        })

    return sub_areas





In [7]:
def run_all(
    config_path: str | Path,
    cities: list[str] | None = None,
    quality_filter: bool = False,
) -> pd.DataFrame:
    """
    Run the full validation pipeline for every city in the AOI tracker CSV.

    Parameters
    ----------
    config_path : str | Path
        Path to ``validation_config.yaml``.
    cities : list[str] | None
        Optional allow-list of ``Dataset code`` values. When supplied, only
        those cities are processed (useful for reruns without editing the CSV).
    quality_filter : bool
        When True, only process rows where ``is_high_quality`` is truthy.

    Returns
    -------
    all_summaries : pd.DataFrame
        City-wide KPI table (one row per city × dataset), also written to
        ``<root_dir>/outputs/metrics/all_cities_summary.{parquet,csv}``.
    """
    logger = _build_logger()
    cfg    = load_config(config_path)

    root      = Path(cfg["root_dir"])
    data_dir  = root / cfg["data_dir"]
    tracker_p = root / cfg["aoi_tracker"]

    if not tracker_p.exists():
        raise FileNotFoundError(f"AOI tracker not found: {tracker_p}")
    if not data_dir.exists():
        raise FileNotFoundError(f"data_dir not found: {data_dir}")

    tracker = load_tracker(tracker_p)
    logger.info("Loaded tracker: %d rows, %d unique Dataset codes",
                len(tracker), tracker["Dataset code"].nunique())

    # filter rows- only suitable AOIs
    tracker = tracker[tracker["Suitable (yes/N)"].str.lower() == "yes"]

    if quality_filter:
        tracker = tracker[tracker["is_high_quality"].apply(_is_truthy)]
        logger.info("Quality filter applied: %d rows remain", len(tracker))

    if cities:
        tracker = tracker[tracker["Dataset code"].isin(cities)]
        logger.info("City filter applied — processing: %s", cities)

    if tracker.empty:
        logger.warning("No rows remain after filtering. Exiting.")
        return pd.DataFrame()

    # ── Process each city ──────────────────────────────────────────────
    all_summaries: list[pd.DataFrame] = []
    failed_cities: list[str]          = []

    grouped = list(tracker.groupby("Dataset code", sort=True))
    logger.info("Processing %d cities.", len(grouped))

    for i, (city, group) in enumerate(grouped, start=1):
        logger.info("=" * 60)
        logger.info("City %d / %d: %s  (%d sub-areas in tracker)",
                    i, len(grouped), city, len(group))
        logger.info("=" * 60)

        try:
            # Resolve file paths for every row in this city's group
            sub_areas = resolve_sub_areas(group, data_dir, logger)

            if not sub_areas:
                logger.warning("%s: no valid sub-areas found. Skipping.", city)
                failed_cities.append(city)
                continue

            # Per-city CRS override: use the first non-null value in the group
            city_cfg = dict(cfg)          # shallow copy — safe for top-level keys
            if "crs" in group.columns:
                crs_vals = group["crs"].dropna().unique()
                if len(crs_vals) == 1:
                    city_cfg["crs"] = str(crs_vals[0])
                    logger.info("  CRS override: %s", city_cfg["crs"])
                elif len(crs_vals) > 1:
                    logger.warning(
                        "  Multiple CRS values in group (%s); using config default %s.",
                        crs_vals, cfg["crs"],
                    )

            v = Validator(city_cfg, city=city, sub_areas=sub_areas, log=logger)
            metrics_all, matches_all = v.compute_vector_metrics()

            if metrics_all.empty:
                logger.warning("%s: no metrics produced, skipping visualisation.", city)
                failed_cities.append(city)
                continue

            summary = v.visualize_metrics(metrics_all, matches_all)
            all_summaries.append(summary)
            logger.info("%s: done ✓", city)

        except Exception:
            logger.exception("Unhandled error processing '%s'. Skipping.", city)
            failed_cities.append(city)

    # ── Combine and save cross-city summary ────────────────────────────
    if not all_summaries:
        logger.error("No cities completed successfully.")
        return pd.DataFrame()

    combined  = pd.concat(all_summaries, ignore_index=True)
    out_dir   = root / "outputs/metrics"
    out_dir.mkdir(parents=True, exist_ok=True)

    combined.to_parquet(out_dir / "all_cities_summary.parquet", index=False)
    combined.to_csv(out_dir / "all_cities_summary.csv",     index=False)
    logger.info("Cross-city summary saved → %s", out_dir)

    if failed_cities:
        logger.warning("Cities with errors or no data: %s", failed_cities)

    logger.info(
        "Done. %d / %d cities completed successfully.",
        len(all_summaries), len(grouped),
    )
    return combined


In [ ]:
config = "configs/validation_configs.yaml" # Path to validation_config.yaml
quality_filter = True
run_all(
        config_path=config,
        quality_filter=quality_filter,
)

2026-03-21 23:50:26 | INFO | Urban_Validator | Loaded tracker: 156 rows, 72 unique Dataset codes
INFO:Urban_Validator:Loaded tracker: 156 rows, 72 unique Dataset codes
2026-03-21 23:50:27 | INFO | Urban_Validator | Quality filter applied: 156 rows remain
INFO:Urban_Validator:Quality filter applied: 156 rows remain
2026-03-21 23:50:27 | INFO | Urban_Validator | Processing 72 cities.
INFO:Urban_Validator:Processing 72 cities.
2026-03-21 23:50:27 | INFO | Urban_Validator | ============================================================
INFO:Urban_Validator:============================================================
2026-03-21 23:50:27 | INFO | Urban_Validator | City 1 / 72: ant-curacao  (1 sub-areas in tracker)
INFO:Urban_Validator:City 1 / 72: ant-curacao  (1 sub-areas in tracker)
2026-03-21 23:50:27 | INFO | Urban_Validator | ============================================================
INFO:Urban_Validator:============================================================
2026-03-21 23:50:27 | 